# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_01 — Multi-Asset Return Engine

### Research question

How do we build a reliable multi-asset return engine that converts prices, FX rates, weights, transaction costs, and contract metadata into clean portfolio returns and PnL attribution?

This notebook opens Phase 4.

Previous phases built data pipelines, pricing models, volatility models, and alpha signals. Phase 4 asks:

> Given signals and instruments, how do we turn them into portfolio returns, exposures, attribution, and risk inputs without hidden look-ahead or accounting mistakes?

This notebook is the foundational accounting layer.

It covers:

1. synthetic multi-asset universe generation;
2. asset master metadata;
3. local prices and FX conversion;
4. adjusted USD return prices;
5. simple and log return calculation;
6. missing-data handling;
7. calendar alignment;
8. portfolio weights;
9. next-period return convention;
10. turnover and transaction costs;
11. portfolio return calculation;
12. dollar PnL calculation;
13. exposure diagnostics;
14. asset-class and currency attribution;
15. futures-style PnL checks;
16. stress-day diagnostics;
17. audit checks;
18. saved outputs and manifest.

The key idea:

> Portfolio research is only as trustworthy as its return engine. If returns, weights, FX, and costs are misaligned, every downstream Sharpe ratio is fiction.

## 1. Why a return engine matters

A return engine is the bridge between:

```text
prices + FX + weights + costs + metadata
```

and:

```text
portfolio returns + PnL + attribution + risk inputs
```

Common failure modes:

1. applying today's weights to today's return after seeing today's close;
2. mixing local and base-currency returns;
3. ignoring FX conversion;
4. using stale prices without flags;
5. treating futures like cash equities;
6. ignoring transaction costs and turnover;
7. losing assets due to calendar mismatches;
8. failing to distinguish price return from total return;
9. accidentally using future-filled data;
10. hiding missing-data assumptions.

This notebook builds a controlled engine and logs the assumptions.

## 2. Return conventions

### Simple return

$$
R_{t+1}=\frac{P_{t+1}}{P_t}-1
$$

### Log return

$$
r_{t+1}=\log(P_{t+1})-\log(P_t)
$$

### Portfolio simple return

If $w_{i,t}$ is the weight decided at time $t$, and $R_{i,t+1}$ is the next-period return:

$$
\begin{aligned}
R^{p}_{t+1} &= \sum_i w_{i,t}R_{i,t+1} \\
&\quad - cost_{t+1}
\end{aligned}
$$

The timing convention matters:

> weights known at $t$ are applied to returns from $t$ to $t+1$.

## 3. FX conversion

For an asset priced in local currency:

$$
P^{USD}_{i,t} = P^{local}_{i,t} \times FX_{ccy,t}
$$

where:

$$
FX_{ccy,t} = \text{USD per 1 unit of local currency}
$$

Then USD return is:

$$
R^{USD}_{i,t+1} = \frac{P^{USD}_{i,t+1}}{P^{USD}_{i,t}}-1
$$

This combines local asset return and currency return.

## 4. Futures and notional exposure

For futures, a price return alone may not equal portfolio return unless weights represent notional exposure.

A simple approximation:

$$
PnL_{i,t+1} = contracts_{i,t} \times multiplier_i \times (P_{i,t+1}-P_{i,t}) \times FX_{i,t+1}
$$

In this notebook, portfolio weights are interpreted as **notional weights**.

The engine also computes a futures-style dollar PnL path to show how contract multipliers enter.

A production futures system needs margin, collateral return, contract rolls, tick values, exchange calendars, and expiry handling.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class ReturnEngineConfig:
    n_days: int = 1_500
    base_currency: str = "USD"
    initial_capital: float = 1_000_000.0
    rebalance_frequency: str = "ME"
    transaction_cost_bps_equity: float = 2.0
    transaction_cost_bps_futures: float = 1.0
    transaction_cost_bps_fx: float = 0.5
    missing_probability: float = 0.002
    stress_probability: float = 0.006
    annualisation: int = 252
    seed: int = 42


config = ReturnEngineConfig()
config

## 6. Asset master

An asset master defines the rules for each instrument.

Fields:

| Field | Meaning |
|---|---|
| `asset` | unique identifier |
| `asset_class` | equity, bond, commodity, FX, crypto |
| `currency` | local pricing currency |
| `multiplier` | contract multiplier or 1 for cash assets |
| `cost_bps` | transaction cost estimate |
| `region` | region grouping |
| `sector` | sector grouping |
| `is_futures` | whether futures-style PnL applies |

A serious backtest should start with metadata, not just price columns.

In [ ]:
def build_asset_master(config: ReturnEngineConfig) -> pd.DataFrame:
    rows = [
        {"asset": "US_EQ", "asset_class": "equity", "currency": "USD", "region": "US", "sector": "equity_index", "multiplier": 1.0, "is_futures": False, "cost_bps": config.transaction_cost_bps_equity},
        {"asset": "EU_EQ", "asset_class": "equity", "currency": "EUR", "region": "Europe", "sector": "equity_index", "multiplier": 1.0, "is_futures": False, "cost_bps": config.transaction_cost_bps_equity},
        {"asset": "JP_EQ", "asset_class": "equity", "currency": "JPY", "region": "Japan", "sector": "equity_index", "multiplier": 1.0, "is_futures": False, "cost_bps": config.transaction_cost_bps_equity},
        {"asset": "US_BOND", "asset_class": "bond", "currency": "USD", "region": "US", "sector": "rates", "multiplier": 1.0, "is_futures": False, "cost_bps": config.transaction_cost_bps_equity},
        {"asset": "EU_BOND", "asset_class": "bond", "currency": "EUR", "region": "Europe", "sector": "rates", "multiplier": 1.0, "is_futures": False, "cost_bps": config.transaction_cost_bps_equity},
        {"asset": "GOLD_FUT", "asset_class": "commodity", "currency": "USD", "region": "Global", "sector": "metals", "multiplier": 100.0, "is_futures": True, "cost_bps": config.transaction_cost_bps_futures},
        {"asset": "OIL_FUT", "asset_class": "commodity", "currency": "USD", "region": "Global", "sector": "energy", "multiplier": 1000.0, "is_futures": True, "cost_bps": config.transaction_cost_bps_futures},
        {"asset": "CN_COPPER_FUT", "asset_class": "commodity", "currency": "CNY", "region": "China", "sector": "metals", "multiplier": 5.0, "is_futures": True, "cost_bps": config.transaction_cost_bps_futures},
        {"asset": "EURUSD_FX", "asset_class": "fx", "currency": "USD", "region": "FX", "sector": "currency", "multiplier": 1.0, "is_futures": False, "cost_bps": config.transaction_cost_bps_fx},
        {"asset": "BTC_USD", "asset_class": "crypto", "currency": "USD", "region": "Global", "sector": "crypto", "multiplier": 1.0, "is_futures": False, "cost_bps": 8.0},
    ]
    return pd.DataFrame(rows)


asset_master = build_asset_master(config)
asset_master

## 7. Synthetic FX rates

FX rates are expressed as:

$$
USD \text{ per 1 local currency unit}
$$

So:

- USD = 1;
- EUR = USD per EUR;
- JPY = USD per JPY;
- CNY = USD per CNY.

This convention makes conversion easy:

$$
P^{USD}=P^{local}\times FX
$$

In [ ]:
def simulate_fx_rates(dates, config: ReturnEngineConfig) -> pd.DataFrame:
    rng = np.random.default_rng(config.seed + 100)

    specs = {
        "USD": {"start": 1.0, "vol": 0.0, "drift": 0.0},
        "EUR": {"start": 1.10, "vol": 0.006, "drift": 0.00000},
        "JPY": {"start": 0.0068, "vol": 0.005, "drift": -0.00001},
        "CNY": {"start": 0.140, "vol": 0.003, "drift": -0.000005},
    }

    rows = []
    n = len(dates)

    for currency, spec in specs.items():
        if currency == "USD":
            rates = np.ones(n)
        else:
            shocks = spec["drift"] + spec["vol"] * rng.standard_normal(n)
            rates = spec["start"] * np.exp(np.cumsum(shocks))

        for date, rate in zip(dates, rates):
            rows.append({"date": date, "currency": currency, "usd_per_local": rate})

    return pd.DataFrame(rows)


dates = pd.bdate_range("2018-01-01", periods=config.n_days)
fx_rates = simulate_fx_rates(dates, config)

fx_rates.head()

In [ ]:
fx_wide = fx_rates.pivot(index="date", columns="currency", values="usd_per_local")

plt.figure(figsize=(12, 6))
for currency in ["EUR", "JPY", "CNY"]:
    plt.plot(fx_wide.index, fx_wide[currency] / fx_wide[currency].iloc[0], label=currency)
plt.title("Synthetic FX Rates, Normalised")
plt.xlabel("Date")
plt.ylabel("Normalised USD per local")
plt.legend()
plt.show()

## 8. Synthetic local prices

We simulate local-currency prices for all assets.

The process includes:

- asset-class-specific drift and volatility;
- correlated shocks;
- occasional stress days;
- missing observations.

Missing observations are inserted deliberately so the return engine can flag and handle them.

In [ ]:
def simulate_local_prices(dates, asset_master, config: ReturnEngineConfig) -> pd.DataFrame:
    rng = np.random.default_rng(config.seed)
    n = len(dates)

    params = {
        "equity": {"drift": 0.00025, "vol": 0.012},
        "bond": {"drift": 0.00008, "vol": 0.004},
        "commodity": {"drift": 0.00010, "vol": 0.018},
        "fx": {"drift": 0.00000, "vol": 0.006},
        "crypto": {"drift": 0.00050, "vol": 0.040},
    }

    market_factor = rng.standard_normal(n)
    rates_factor = rng.standard_normal(n)
    commodity_factor = rng.standard_normal(n)
    stress_indicator = rng.uniform(size=n) < config.stress_probability

    start_prices = {
        "US_EQ": 100.0,
        "EU_EQ": 90.0,
        "JP_EQ": 12000.0,
        "US_BOND": 100.0,
        "EU_BOND": 100.0,
        "GOLD_FUT": 1900.0,
        "OIL_FUT": 75.0,
        "CN_COPPER_FUT": 68000.0,
        "EURUSD_FX": 1.10,
        "BTC_USD": 30000.0,
    }

    rows = []

    for _, meta in asset_master.iterrows():
        asset = meta["asset"]
        asset_class = meta["asset_class"]
        p = params[asset_class]

        shocks = p["drift"] + p["vol"] * rng.standard_normal(n)

        if asset_class == "equity":
            shocks += 0.006 * market_factor
            shocks[stress_indicator] -= rng.uniform(0.015, 0.050, size=np.sum(stress_indicator))
        elif asset_class == "bond":
            shocks += -0.003 * rates_factor
            shocks[stress_indicator] += rng.uniform(0.002, 0.012, size=np.sum(stress_indicator))
        elif asset_class == "commodity":
            shocks += 0.008 * commodity_factor + 0.002 * market_factor
            shocks[stress_indicator] += rng.normal(0.0, 0.030, size=np.sum(stress_indicator))
        elif asset_class == "crypto":
            shocks += 0.010 * market_factor
            shocks[stress_indicator] -= rng.uniform(0.030, 0.100, size=np.sum(stress_indicator))
        elif asset_class == "fx":
            shocks += 0.002 * market_factor

        true_price = start_prices[asset] * np.exp(np.cumsum(shocks))

        missing = rng.uniform(size=n) < config.missing_probability
        observed_price = true_price.copy()
        observed_price[missing] = np.nan

        for date, px, true_px, is_stress in zip(dates, observed_price, true_price, stress_indicator):
            rows.append({
                "date": date,
                "asset": asset,
                "local_price": px,
                "true_local_price_before_missing": true_px,
                "stress_day": bool(is_stress),
            })

    return pd.DataFrame(rows)


local_prices = simulate_local_prices(dates, asset_master, config)
local_prices.head()

## 9. Local price panel diagnostics

Before calculating returns, inspect missingness.

A return engine should not silently hide missing data.

In [ ]:
missing_report = (
    local_prices
    .groupby("asset")
    .agg(
        n_obs=("local_price", "size"),
        n_missing=("local_price", lambda x: x.isna().sum()),
        missing_rate=("local_price", lambda x: x.isna().mean())
    )
    .reset_index()
    .sort_values("missing_rate", ascending=False)
)

missing_report

In [ ]:
local_price_wide = local_prices.pivot(index="date", columns="asset", values="local_price")

plt.figure(figsize=(12, 6))
for asset in ["US_EQ", "EU_EQ", "US_BOND", "GOLD_FUT", "OIL_FUT", "BTC_USD"]:
    series = local_price_wide[asset].dropna()
    plt.plot(series.index, series / series.iloc[0], label=asset)
plt.title("Synthetic Local Prices, Normalised")
plt.xlabel("Date")
plt.ylabel("Normalised local price")
plt.legend()
plt.show()

## 10. Merge metadata and FX

We join:

```text
local_prices + asset_master + fx_rates
```

Then compute USD price:

$$
P^{USD}_{i,t}=P^{local}_{i,t}\times FX_{ccy,t}
$$

In [ ]:
def attach_metadata_and_fx(local_prices, asset_master, fx_rates):
    out = local_prices.merge(asset_master, on="asset", how="left")
    out = out.merge(fx_rates, on=["date", "currency"], how="left")

    out["usd_per_local"] = out["usd_per_local"].fillna(1.0)
    out["usd_price_raw"] = out["local_price"] * out["usd_per_local"]
    out["true_usd_price_before_missing"] = out["true_local_price_before_missing"] * out["usd_per_local"]

    return out


price_panel_raw = attach_metadata_and_fx(local_prices, asset_master, fx_rates)
price_panel_raw.head()

## 11. Missing-data policy

This notebook uses a conservative policy:

1. forward-fill missing prices within each asset;
2. flag prices that were filled;
3. prevent returns from being treated as fresh when either endpoint is filled;
4. keep audit columns.

A production system may instead drop stale assets, carry stale returns as zero, or use exchange-specific calendars.

The policy must be explicit.

In [ ]:
def apply_missing_data_policy(panel):
    out = panel.sort_values(["asset", "date"]).copy()

    out["local_price_was_missing"] = out["local_price"].isna()
    out["usd_price"] = out.groupby("asset")["usd_price_raw"].ffill()
    out["usd_price_was_filled"] = out["usd_price_raw"].isna() & out["usd_price"].notna()

    # Backfill only for initial rows where an asset begins with missing data, and flag it.
    out["usd_price"] = out.groupby("asset")["usd_price"].bfill()
    out["usd_price_initial_backfill"] = (
        out["usd_price_raw"].isna()
        & out["usd_price"].notna()
        & ~out["usd_price_was_filled"]
    )

    return out


price_panel = apply_missing_data_policy(price_panel_raw)
price_panel.head()

In [ ]:
fill_report = (
    price_panel
    .groupby("asset")
    .agg(
        filled_rate=("usd_price_was_filled", "mean"),
        initial_backfill_rate=("usd_price_initial_backfill", "mean"),
        raw_missing_rate=("local_price_was_missing", "mean")
    )
    .reset_index()
)

fill_report

## 12. Return calculation

For each asset, calculate:

### Simple USD return

$$
R_{i,t} = \frac{P^{USD}_{i,t}}{P^{USD}_{i,t-1}}-1
$$

### Log USD return

$$
r_{i,t} = \log P^{USD}_{i,t}-\log P^{USD}_{i,t-1}
$$

We also flag stale returns where current or previous price was filled.

In [ ]:
def calculate_returns(panel):
    out = panel.sort_values(["asset", "date"]).copy()

    out["usd_price_lag1"] = out.groupby("asset")["usd_price"].shift(1)
    out["simple_return_usd"] = out["usd_price"] / out["usd_price_lag1"] - 1.0
    out["log_return_usd"] = np.log(out["usd_price"] / out["usd_price_lag1"])

    out["current_price_filled"] = out["usd_price_was_filled"] | out["usd_price_initial_backfill"]
    out["previous_price_filled"] = out.groupby("asset")["current_price_filled"].shift(1).fillna(False)
    out["return_has_stale_endpoint"] = out["current_price_filled"] | out["previous_price_filled"]

    out["simple_return_usd"] = out["simple_return_usd"].replace([np.inf, -np.inf], np.nan)
    out["log_return_usd"] = out["log_return_usd"].replace([np.inf, -np.inf], np.nan)

    return out


return_panel = calculate_returns(price_panel)

return_panel[[
    "date", "asset", "usd_price", "simple_return_usd", "log_return_usd", "return_has_stale_endpoint"
]].head(12)

## 13. Return matrix

Many portfolio operations require a wide return matrix:

$$
R_{t,i}
$$

Rows are dates, columns are assets.

The engine keeps both panel and matrix forms.

In [ ]:
return_matrix = return_panel.pivot(index="date", columns="asset", values="simple_return_usd").sort_index()
log_return_matrix = return_panel.pivot(index="date", columns="asset", values="log_return_usd").sort_index()
stale_return_matrix = return_panel.pivot(index="date", columns="asset", values="return_has_stale_endpoint").sort_index()

# Fill first return with zero for portfolio path initialisation.
return_matrix_clean = return_matrix.fillna(0.0)

return_matrix_clean.head()

In [ ]:
return_summary = (
    return_matrix
    .agg(["mean", "std", "min", "max"])
    .T
    .rename(columns={
        "mean": "daily_mean",
        "std": "daily_vol",
        "min": "min_daily_return",
        "max": "max_daily_return"
    })
)

return_summary["annualised_mean"] = return_summary["daily_mean"] * config.annualisation
return_summary["annualised_vol"] = return_summary["daily_vol"] * np.sqrt(config.annualisation)

return_summary.sort_values("annualised_vol", ascending=False)

## 14. Asset-class return diagnostics

Group returns by asset class.

This helps catch mistakes such as:

- crypto volatility looking like bonds;
- bond returns accidentally scaled by 100;
- JPY conversion error;
- futures multipliers mistakenly applied to percentage returns.

In [ ]:
asset_class_map = asset_master.set_index("asset")["asset_class"].to_dict()

class_return_summary = (
    return_summary
    .assign(asset_class=lambda x: x.index.map(asset_class_map))
    .groupby("asset_class")
    .agg(
        mean_annualised_vol=("annualised_vol", "mean"),
        max_annualised_vol=("annualised_vol", "max"),
        mean_annualised_mean=("annualised_mean", "mean")
    )
    .reset_index()
    .sort_values("mean_annualised_vol", ascending=False)
)

class_return_summary

In [ ]:
plt.figure(figsize=(10, 6))
vol_plot = return_summary.sort_values("annualised_vol")
plt.barh(vol_plot.index, vol_plot["annualised_vol"])
plt.title("Annualised Volatility by Asset")
plt.xlabel("Annualised volatility")
plt.ylabel("Asset")
plt.show()

## 15. Generate synthetic target weights

We create a monthly-rebalanced multi-asset portfolio.

Rules:

- long-only core allocation;
- risk-aware tilt away from high volatility;
- small tactical signal tilt;
- weights sum to 1;
- weights are decided at close $t$ and applied to returns $t+1$.

This is not an optimal portfolio. It is a return-engine test case.

In [ ]:
def generate_target_weights(return_matrix, asset_master, config):
    dates = return_matrix.index
    assets = return_matrix.columns.tolist()

    base_class_weights = {
        "equity": 0.35,
        "bond": 0.25,
        "commodity": 0.20,
        "fx": 0.05,
        "crypto": 0.15,
    }

    asset_classes = asset_master.set_index("asset")["asset_class"]

    base = pd.Series(0.0, index=assets)
    for asset_class, weight in base_class_weights.items():
        class_assets = [a for a in assets if asset_classes[a] == asset_class]
        if class_assets:
            base.loc[class_assets] = weight / len(class_assets)

    month_ends = pd.Series(dates, index=dates).groupby(pd.Grouper(freq=config.rebalance_frequency)).last().dropna()
    rebalance_dates = set(pd.to_datetime(month_ends.values))

    rolling_vol = return_matrix.rolling(63, min_periods=20).std()
    trailing_return = (1 + return_matrix.fillna(0)).rolling(63, min_periods=20).apply(lambda x: np.prod(x) - 1, raw=True)

    rows = []
    current_weights = base.copy()

    for date in dates:
        if pd.Timestamp(date) in rebalance_dates:
            vol = rolling_vol.loc[date].replace(0, np.nan)
            inv_vol = 1.0 / vol
            inv_vol = inv_vol.replace([np.inf, -np.inf], np.nan).fillna(inv_vol.median())
            inv_vol_weights = inv_vol / inv_vol.sum()

            momentum = trailing_return.loc[date].fillna(0.0)
            signal = momentum.rank(pct=True) - 0.5

            raw = 0.70 * base + 0.25 * inv_vol_weights + 0.05 * signal
            raw = raw.clip(lower=0.0)

            if raw.sum() <= 0:
                current_weights = base.copy()
            else:
                current_weights = raw / raw.sum()

        rows.append(pd.Series(current_weights, name=date))

    weights = pd.DataFrame(rows)
    weights.index.name = "date"
    return weights


target_weights = generate_target_weights(return_matrix_clean, asset_master, config)

target_weights.head(), target_weights.sum(axis=1).describe()

In [ ]:
plt.figure(figsize=(12, 6))
for asset in target_weights.columns:
    plt.plot(target_weights.index, target_weights[asset], label=asset, alpha=0.85)
plt.title("Target Weights")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.legend(ncol=2)
plt.show()

## 16. Return timing convention

The correct portfolio return convention is:

$$
R^p_t = \sum_i w_{i,t-1}R_{i,t}
$$

So we shift weights forward relative to returns.

Implementation:

```text
weights_at_t_minus_1 = target_weights.shift(1)
portfolio_return_at_t = sum(weights_at_t_minus_1 * asset_return_at_t)
```

The first day is set to zero because no prior weights exist.

In [ ]:
def align_weights_to_returns(weights, returns):
    aligned = weights.reindex(returns.index).ffill()
    investable_weights = aligned.shift(1).fillna(0.0)
    return investable_weights


investable_weights = align_weights_to_returns(target_weights, return_matrix_clean)
investable_weights.head()

## 17. Turnover and transaction costs

Turnover:

$$
TO_t = \sum_i |w_{i,t}-w_{i,t-1}|
$$

Transaction cost:

$$
cost_t = \sum_i |w_{i,t}-w_{i,t-1}| \times costbps_i / 10000
$$

Costs are deducted from portfolio return.

This is a simple proportional-cost model.

In [ ]:
def calculate_turnover_and_costs(weights, asset_master):
    trade_weights = weights.diff().abs().fillna(weights.abs())

    cost_bps = asset_master.set_index("asset").reindex(weights.columns)["cost_bps"].fillna(2.0)
    cost_rates = cost_bps / 10000.0

    cost_by_asset = trade_weights.multiply(cost_rates, axis=1)
    total_cost = cost_by_asset.sum(axis=1)
    turnover = trade_weights.sum(axis=1)

    return trade_weights, cost_by_asset, total_cost, turnover


trade_weights, cost_by_asset, total_cost, turnover = calculate_turnover_and_costs(investable_weights, asset_master)

pd.DataFrame({
    "turnover": turnover,
    "transaction_cost_return": total_cost
}).head()

## 18. Portfolio return calculation

Gross return:

$$
R^{gross}_t = \sum_i w_{i,t-1}R_{i,t}
$$

Net return:

$$
R^{net}_t = R^{gross}_t - cost_t
$$

Cumulative NAV:

$$
NAV_t = NAV_0 \prod_{\tau \le t}(1+R^{net}_\tau)
$$

In [ ]:
def calculate_portfolio_returns(investable_weights, returns, total_cost, initial_capital):
    gross_asset_contrib = investable_weights * returns
    gross_return = gross_asset_contrib.sum(axis=1)

    aligned_cost = total_cost.reindex(gross_return.index).fillna(0.0)
    net_return = gross_return - aligned_cost
    nav = initial_capital * (1 + net_return).cumprod()

    out = pd.DataFrame({
        "gross_return": gross_return,
        "transaction_cost_return": aligned_cost,
        "net_return": net_return,
        "nav": nav
    })

    return out, gross_asset_contrib


portfolio_returns, asset_contributions = calculate_portfolio_returns(
    investable_weights,
    return_matrix_clean,
    total_cost,
    config.initial_capital
)

portfolio_returns.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(portfolio_returns.index, portfolio_returns["nav"])
plt.title("Portfolio NAV")
plt.xlabel("Date")
plt.ylabel("NAV")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(portfolio_returns.index, portfolio_returns["gross_return"], label="gross", alpha=0.7)
plt.plot(portfolio_returns.index, portfolio_returns["net_return"], label="net", alpha=0.7)
plt.axhline(0, linestyle="--")
plt.title("Daily Portfolio Returns")
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.show()

## 19. Portfolio risk summary

Compute:

- annualised return;
- annualised volatility;
- Sharpe ratio proxy;
- max drawdown;
- cost drag;
- average turnover.

In [ ]:
def max_drawdown(nav):
    running_max = nav.cummax()
    drawdown = nav / running_max - 1.0
    return drawdown, drawdown.min()


def portfolio_summary(portfolio_returns, turnover, config):
    r = portfolio_returns["net_return"]
    nav = portfolio_returns["nav"]
    drawdown, mdd = max_drawdown(nav)

    return pd.Series({
        "annualised_return": float(r.mean() * config.annualisation),
        "annualised_vol": float(r.std(ddof=1) * np.sqrt(config.annualisation)),
        "sharpe_proxy": float(r.mean() / r.std(ddof=1) * np.sqrt(config.annualisation)) if r.std(ddof=1) > 0 else np.nan,
        "max_drawdown": float(mdd),
        "total_return": float(nav.iloc[-1] / nav.iloc[0] - 1.0),
        "mean_daily_cost": float(portfolio_returns["transaction_cost_return"].mean()),
        "annualised_cost_drag": float(portfolio_returns["transaction_cost_return"].mean() * config.annualisation),
        "mean_daily_turnover": float(turnover.mean()),
        "p95_daily_turnover": float(turnover.quantile(0.95)),
    })


portfolio_stats = portfolio_summary(portfolio_returns, turnover, config)
portfolio_stats

In [ ]:
drawdown, _ = max_drawdown(portfolio_returns["nav"])

plt.figure(figsize=(12, 5))
plt.plot(drawdown.index, drawdown)
plt.title("Portfolio Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.show()

## 20. Dollar PnL

Dollar PnL:

$$
PnL_t = NAV_{t-1}R^{net}_t
$$

Asset contribution in dollars:

$$
PnL_{i,t} = NAV_{t-1}w_{i,t-1}R_{i,t}
$$

This supports attribution.

In [ ]:
def calculate_dollar_pnl(portfolio_returns, asset_contributions, initial_capital):
    nav_lag = portfolio_returns["nav"].shift(1).fillna(initial_capital)

    asset_pnl = asset_contributions.multiply(nav_lag, axis=0)
    gross_pnl = asset_pnl.sum(axis=1)
    cost_pnl = portfolio_returns["transaction_cost_return"] * nav_lag
    net_pnl = portfolio_returns["net_return"] * nav_lag

    pnl = pd.DataFrame({
        "gross_pnl": gross_pnl,
        "cost_pnl": cost_pnl,
        "net_pnl": net_pnl
    })

    return pnl, asset_pnl


pnl_table, asset_pnl = calculate_dollar_pnl(portfolio_returns, asset_contributions, config.initial_capital)

pnl_table.head()

## 21. Asset attribution

Total asset contribution:

$$
\sum_t PnL_{i,t}
$$

We rank assets by their contribution to portfolio PnL.

In [ ]:
asset_attribution = asset_pnl.sum().rename("total_gross_pnl_contribution").to_frame()
asset_attribution["asset_class"] = asset_attribution.index.map(asset_master.set_index("asset")["asset_class"].to_dict())
asset_attribution["currency"] = asset_attribution.index.map(asset_master.set_index("asset")["currency"].to_dict())
asset_attribution = asset_attribution.sort_values("total_gross_pnl_contribution", ascending=False)

asset_attribution

In [ ]:
plt.figure(figsize=(10, 6))
attr_plot = asset_attribution.sort_values("total_gross_pnl_contribution")
plt.barh(attr_plot.index, attr_plot["total_gross_pnl_contribution"])
plt.title("Asset PnL Attribution")
plt.xlabel("Total gross PnL contribution")
plt.ylabel("Asset")
plt.show()

## 22. Asset-class attribution

Group asset PnL by asset class:

$$
PnL_{class,t}=\sum_{i\in class}PnL_{i,t}
$$

In [ ]:
asset_class_by_asset = asset_master.set_index("asset")["asset_class"].to_dict()

class_pnl_daily = pd.DataFrame(index=asset_pnl.index)
for asset_class in sorted(asset_master["asset_class"].unique()):
    class_assets = [asset for asset in asset_pnl.columns if asset_class_by_asset[asset] == asset_class]
    class_pnl_daily[asset_class] = asset_pnl[class_assets].sum(axis=1)

class_attribution = class_pnl_daily.sum().rename("total_gross_pnl").to_frame()
class_attribution["mean_daily_pnl"] = class_pnl_daily.mean()
class_attribution["pnl_vol"] = class_pnl_daily.std()

class_attribution.sort_values("total_gross_pnl", ascending=False)

In [ ]:
plt.figure(figsize=(12, 6))
for asset_class in class_pnl_daily.columns:
    plt.plot(class_pnl_daily.index, class_pnl_daily[asset_class].cumsum(), label=asset_class)
plt.axhline(0, linestyle="--")
plt.title("Cumulative Asset-Class PnL")
plt.xlabel("Date")
plt.ylabel("Cumulative gross PnL")
plt.legend()
plt.show()

## 23. Currency attribution

For non-USD assets, USD return combines:

1. local asset return;
2. FX return;
3. interaction term.

Approximation:

$$
R^{USD} \approx R^{local}+R^{FX}
$$

This section computes currency grouping of PnL, not full Brinson-style FX attribution.

In [ ]:
currency_by_asset = asset_master.set_index("asset")["currency"].to_dict()

currency_pnl_daily = pd.DataFrame(index=asset_pnl.index)
for currency in sorted(asset_master["currency"].unique()):
    currency_assets = [asset for asset in asset_pnl.columns if currency_by_asset[asset] == currency]
    currency_pnl_daily[currency] = asset_pnl[currency_assets].sum(axis=1)

currency_attribution = currency_pnl_daily.sum().rename("total_gross_pnl").to_frame()
currency_attribution["mean_daily_pnl"] = currency_pnl_daily.mean()
currency_attribution["pnl_vol"] = currency_pnl_daily.std()

currency_attribution.sort_values("total_gross_pnl", ascending=False)

## 24. Exposure diagnostics

Portfolio diagnostics:

- gross exposure;
- net exposure;
- asset-class exposure;
- currency exposure;
- futures notional exposure.

For long-only weights, gross and net are both near 1.

For long/short portfolios, they can differ sharply.

In [ ]:
def exposure_diagnostics(weights, asset_master):
    gross_exposure = weights.abs().sum(axis=1)
    net_exposure = weights.sum(axis=1)

    class_map = asset_master.set_index("asset")["asset_class"].to_dict()
    currency_map = asset_master.set_index("asset")["currency"].to_dict()
    futures_map = asset_master.set_index("asset")["is_futures"].to_dict()

    class_exposure = pd.DataFrame(index=weights.index)
    for asset_class in sorted(asset_master["asset_class"].unique()):
        assets = [asset for asset in weights.columns if class_map[asset] == asset_class]
        class_exposure[asset_class] = weights[assets].sum(axis=1)

    currency_exposure = pd.DataFrame(index=weights.index)
    for currency in sorted(asset_master["currency"].unique()):
        assets = [asset for asset in weights.columns if currency_map[asset] == currency]
        currency_exposure[currency] = weights[assets].sum(axis=1)

    futures_assets = [asset for asset in weights.columns if futures_map[asset]]
    futures_abs_notional_exposure = weights[futures_assets].abs().sum(axis=1) if futures_assets else pd.Series(0.0, index=weights.index)

    summary = pd.DataFrame({
        "gross_exposure": gross_exposure,
        "net_exposure": net_exposure,
        "futures_abs_notional_exposure": futures_abs_notional_exposure
    })

    return summary, class_exposure, currency_exposure


exposure_summary, class_exposure, currency_exposure = exposure_diagnostics(investable_weights, asset_master)

exposure_summary.describe()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(exposure_summary.index, exposure_summary["gross_exposure"], label="gross")
plt.plot(exposure_summary.index, exposure_summary["net_exposure"], label="net")
plt.plot(exposure_summary.index, exposure_summary["futures_abs_notional_exposure"], label="futures abs notional")
plt.title("Portfolio Exposure Diagnostics")
plt.xlabel("Date")
plt.ylabel("Exposure")
plt.legend()
plt.show()

## 25. Futures-style PnL check

For futures assets, a futures-style dollar PnL uses:

$$
contracts_t = \frac{NAV_t \times weight_t} {price_t \times multiplier \times FX_t}
$$

Then:

$$
PnL_{t+1} = contracts_t \times multiplier \times (P_{t+1}-P_t) \times FX_{t+1}
$$

This section compares futures-style PnL with the notional-return approximation.

In [ ]:
def futures_style_pnl_check(price_panel, weights, portfolio_returns, asset_pnl, asset_master, initial_capital):
    futures_assets = asset_master.loc[asset_master["is_futures"], "asset"].tolist()
    if not futures_assets:
        return pd.DataFrame()

    local_price = price_panel.pivot(index="date", columns="asset", values="local_price").reindex(weights.index).ffill()
    fx = price_panel.pivot(index="date", columns="asset", values="usd_per_local").reindex(weights.index).ffill()
    multipliers = asset_master.set_index("asset")["multiplier"]

    nav_lag = portfolio_returns["nav"].shift(1).fillna(initial_capital).reindex(weights.index)

    rows = []

    for asset in futures_assets:
        px = local_price[asset]
        fx_rate = fx[asset]
        multiplier = multipliers.loc[asset]
        w = weights[asset]

        notional_per_contract = px * multiplier * fx_rate
        contracts = (nav_lag * w) / notional_per_contract.replace(0, np.nan)

        price_change = px.diff()
        pnl = contracts.shift(1).fillna(0.0) * multiplier * price_change * fx_rate
        return_approx_pnl = asset_pnl[asset].reindex(weights.index).fillna(0.0)

        rows.append(pd.DataFrame({
            "date": weights.index,
            "asset": asset,
            "contracts": contracts,
            "futures_style_pnl": pnl,
            "notional_return_approx_pnl": return_approx_pnl,
            "difference": pnl - return_approx_pnl
        }))

    return pd.concat(rows, ignore_index=True)


futures_pnl_check = futures_style_pnl_check(
    price_panel,
    investable_weights,
    portfolio_returns,
    asset_pnl,
    asset_master,
    config.initial_capital
)

futures_pnl_check.head()

In [ ]:
futures_pnl_summary = (
    futures_pnl_check
    .groupby("asset")
    .agg(
        mean_abs_difference=("difference", lambda x: np.mean(np.abs(x))),
        total_futures_style_pnl=("futures_style_pnl", "sum"),
        total_return_approx_pnl=("notional_return_approx_pnl", "sum")
    )
    .reset_index()
)

futures_pnl_summary

## 26. Stress-day diagnostics

The synthetic data includes stress days.

We compare portfolio behaviour on stress and non-stress days.

In [ ]:
stress_by_date = (
    price_panel
    .groupby("date")["stress_day"]
    .max()
    .reindex(portfolio_returns.index)
    .fillna(False)
)

stress_report = pd.DataFrame({
    "stress_day": stress_by_date,
    "net_return": portfolio_returns["net_return"],
    "gross_return": portfolio_returns["gross_return"],
    "transaction_cost_return": portfolio_returns["transaction_cost_return"],
    "turnover": turnover.reindex(portfolio_returns.index)
})

stress_summary = (
    stress_report
    .groupby("stress_day")
    .agg(
        n_days=("net_return", "count"),
        mean_net_return=("net_return", "mean"),
        vol_net_return=("net_return", "std"),
        worst_net_return=("net_return", "min"),
        mean_turnover=("turnover", "mean"),
        mean_cost=("transaction_cost_return", "mean")
    )
    .reset_index()
)

stress_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(stress_report.loc[~stress_report["stress_day"], "net_return"], bins=80, alpha=0.6, density=True, label="normal")
plt.hist(stress_report.loc[stress_report["stress_day"], "net_return"], bins=40, alpha=0.6, density=True, label="stress")
plt.axvline(0, linestyle="--")
plt.title("Portfolio Returns: Stress vs Normal Days")
plt.xlabel("Net return")
plt.ylabel("Density")
plt.legend()
plt.show()

## 27. Audit checks

A return engine should fail loudly when basic accounting breaks.

Checks:

1. weights sum near target;
2. returns are finite;
3. no unexpected missing returns;
4. costs are non-negative;
5. NAV path is positive;
6. stale-return rate is reported;
7. no future-weight alignment.

In [ ]:
def run_audit_checks(target_weights, investable_weights, return_matrix, portfolio_returns, total_cost, stale_return_matrix):
    checks = []

    weight_sum_error = (target_weights.sum(axis=1) - 1.0).abs().max()
    checks.append({
        "check": "target_weights_sum_to_one",
        "value": float(weight_sum_error),
        "passed": bool(weight_sum_error < 1e-8)
    })

    first_day_investable_sum = investable_weights.iloc[0].sum()
    checks.append({
        "check": "first_day_investable_weights_zero",
        "value": float(first_day_investable_sum),
        "passed": bool(abs(first_day_investable_sum) < 1e-12)
    })

    finite_returns = np.isfinite(return_matrix.fillna(0.0).to_numpy()).all()
    checks.append({
        "check": "returns_finite_after_cleaning",
        "value": bool(finite_returns),
        "passed": bool(finite_returns)
    })

    costs_nonnegative = (total_cost.fillna(0.0) >= -1e-12).all()
    checks.append({
        "check": "transaction_costs_nonnegative",
        "value": bool(costs_nonnegative),
        "passed": bool(costs_nonnegative)
    })

    nav_positive = (portfolio_returns["nav"] > 0).all()
    checks.append({
        "check": "nav_positive",
        "value": bool(nav_positive),
        "passed": bool(nav_positive)
    })

    stale_rate = stale_return_matrix.fillna(False).mean().mean()
    checks.append({
        "check": "stale_return_endpoint_rate_reported",
        "value": float(stale_rate),
        "passed": bool(stale_rate < 0.05)
    })

    return pd.DataFrame(checks)


audit_report = run_audit_checks(
    target_weights,
    investable_weights,
    return_matrix,
    portfolio_returns,
    total_cost,
    stale_return_matrix
)

audit_report

## 28. Output schemas

The engine produces several reusable tables.

### `return_panel`

Long-form asset return table.

### `return_matrix`

Wide asset return matrix.

### `target_weights`

Portfolio weights before timing shift.

### `investable_weights`

Weights shifted to avoid look-ahead.

### `portfolio_returns`

Gross/net returns and NAV.

### `asset_pnl`

Daily asset-level PnL attribution.

### `audit_report`

Return-engine checks.

In [ ]:
schema_report = pd.DataFrame([
    {"table": "asset_master", "rows": len(asset_master), "columns": ", ".join(asset_master.columns)},
    {"table": "fx_rates", "rows": len(fx_rates), "columns": ", ".join(fx_rates.columns)},
    {"table": "price_panel", "rows": len(price_panel), "columns": ", ".join(price_panel.columns[:12]) + "..."},
    {"table": "return_panel", "rows": len(return_panel), "columns": ", ".join(return_panel.columns[:12]) + "..."},
    {"table": "return_matrix", "rows": len(return_matrix), "columns": f"{return_matrix.shape[1]} assets"},
    {"table": "target_weights", "rows": len(target_weights), "columns": f"{target_weights.shape[1]} assets"},
    {"table": "portfolio_returns", "rows": len(portfolio_returns), "columns": ", ".join(portfolio_returns.columns)},
    {"table": "asset_pnl", "rows": len(asset_pnl), "columns": f"{asset_pnl.shape[1]} assets"},
    {"table": "audit_report", "rows": len(audit_report), "columns": ", ".join(audit_report.columns)}
])

schema_report

## 29. Saving outputs

The notebook saves:

1. asset master;
2. FX rates;
3. local prices;
4. cleaned price panel;
5. return panel;
6. return matrix;
7. target weights;
8. investable weights;
9. trades and costs;
10. portfolio returns;
11. asset PnL;
12. attribution reports;
13. exposure diagnostics;
14. futures PnL check;
15. stress diagnostics;
16. audit report;
17. manifest.

In [ ]:
output_dir = Path("data/processed/multi_asset_return_engine_v1")
output_dir.mkdir(parents=True, exist_ok=True)

asset_master_path = output_dir / "asset_master.csv"
fx_rates_path = output_dir / "fx_rates.csv"
local_prices_path = output_dir / "local_prices.csv"
price_panel_path = output_dir / "clean_price_panel.csv"
return_panel_path = output_dir / "return_panel.csv"
return_matrix_path = output_dir / "return_matrix.csv"
target_weights_path = output_dir / "target_weights.csv"
investable_weights_path = output_dir / "investable_weights.csv"
trade_weights_path = output_dir / "trade_weights.csv"
cost_by_asset_path = output_dir / "cost_by_asset.csv"
portfolio_returns_path = output_dir / "portfolio_returns.csv"
pnl_table_path = output_dir / "pnl_table.csv"
asset_pnl_path = output_dir / "asset_pnl.csv"
asset_attribution_path = output_dir / "asset_attribution.csv"
class_attribution_path = output_dir / "class_attribution.csv"
currency_attribution_path = output_dir / "currency_attribution.csv"
exposure_summary_path = output_dir / "exposure_summary.csv"
class_exposure_path = output_dir / "class_exposure.csv"
currency_exposure_path = output_dir / "currency_exposure.csv"
futures_pnl_check_path = output_dir / "futures_pnl_check.csv"
stress_summary_path = output_dir / "stress_summary.csv"
audit_report_path = output_dir / "audit_report.csv"
schema_report_path = output_dir / "schema_report.csv"
manifest_path = output_dir / "manifest.json"

asset_master.to_csv(asset_master_path, index=False)
fx_rates.to_csv(fx_rates_path, index=False)
local_prices.to_csv(local_prices_path, index=False)
price_panel.to_csv(price_panel_path, index=False)
return_panel.to_csv(return_panel_path, index=False)
return_matrix.to_csv(return_matrix_path)
target_weights.to_csv(target_weights_path)
investable_weights.to_csv(investable_weights_path)
trade_weights.to_csv(trade_weights_path)
cost_by_asset.to_csv(cost_by_asset_path)
portfolio_returns.to_csv(portfolio_returns_path)
pnl_table.to_csv(pnl_table_path)
asset_pnl.to_csv(asset_pnl_path)
asset_attribution.to_csv(asset_attribution_path)
class_attribution.to_csv(class_attribution_path)
currency_attribution.to_csv(currency_attribution_path)
exposure_summary.to_csv(exposure_summary_path)
class_exposure.to_csv(class_exposure_path)
currency_exposure.to_csv(currency_exposure_path)
futures_pnl_check.to_csv(futures_pnl_check_path, index=False)
stress_summary.to_csv(stress_summary_path, index=False)
audit_report.to_csv(audit_report_path, index=False)
schema_report.to_csv(schema_report_path, index=False)

manifest = {
    "dataset_name": "multi_asset_return_engine_outputs",
    "schema_version": "multi_asset_return_engine_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "base_currency": config.base_currency,
    "n_assets": int(asset_master["asset"].nunique()),
    "date_range": {
        "start": str(return_matrix.index.min()),
        "end": str(return_matrix.index.max())
    },
    "portfolio_stats": portfolio_stats.to_dict(),
    "audit_passed": bool(audit_report["passed"].all()),
    "audit_report": audit_report.to_dict(orient="records"),
    "tables": schema_report.to_dict(orient="records"),
    "notes": [
        "Weights are shifted by one day before applying returns to avoid look-ahead.",
        "Prices are converted to USD using USD-per-local FX convention.",
        "Missing prices are forward-filled and stale return endpoints are flagged.",
        "Transaction costs are proportional to turnover using asset-level bps estimates.",
        "Futures PnL check compares notional-return approximation with contract multiplier calculation.",
        "This return engine is synthetic and educational; production systems need exchange calendars, rolls, corporate actions, and point-in-time data controls."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

asset_master_path, return_panel_path, portfolio_returns_path, audit_report_path, manifest_path

## 30. Limitations

### 30.1 Synthetic data

The notebook uses synthetic prices and FX rates.

Real data requires vendor-specific cleaning, corporate actions, contract rolls, and exchange calendars.

### 30.2 Simplified missing-data policy

Forward-fill is not always appropriate.

Some assets should be marked untradable when stale.

### 30.3 Simplified transaction costs

Costs are fixed bps by asset.

Real costs depend on liquidity, volatility, spread, participation rate, market impact, and venue.

### 30.4 Futures simplification

Futures treatment is simplified.

A full system requires:

- expiry and roll schedules;
- contract multipliers;
- tick sizes;
- margin;
- collateral return;
- continuous contract construction;
- position limits.

### 30.5 No benchmark-relative accounting

The notebook computes absolute returns, not benchmark-relative active returns.

### 30.6 No tax, financing, or borrow

Financing, short borrow, margin interest, and taxes are ignored.

### 30.7 No intraday execution

Weights rebalance at daily frequency.

Real portfolios need execution timing and fill models.

## 31. Research frontier and extensions

Important extensions include:

1. **Point-in-time asset master**  
   Track metadata changes through time.

2. **Corporate actions and total return**  
   Dividends, splits, distributions, coupon accruals.

3. **Futures roll engine**  
   Contract selection, roll dates, adjusted continuous series.

4. **Collateral and cash accounting**  
   Include cash return and margin collateral.

5. **FX hedging layer**  
   Separate local asset return and currency overlay return.

6. **Transaction-cost model**  
   Spread, slippage, impact, participation rate.

7. **Intraday execution model**  
   VWAP, TWAP, arrival price, and execution slippage.

8. **Benchmark-relative return engine**  
   Active weights, active returns, and tracking error.

9. **Multi-currency portfolio accounting**  
   Local books, base-currency book, and FX translation effects.

10. **Chinese futures extension**  
   Contract multipliers, night sessions, price limits, exchange calendars, and main-contract rolls.

## 32. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_02_covariance_estimation_and_shrinkage**  
   Turn return matrices into risk estimates.

2. **04_03_risk_model_factor_exposures**  
   Decompose returns into factor exposures.

3. **04_04_mean_variance_optimization_constraints**  
   Use returns and covariance for allocation.

4. **04_05_risk_parity_and_volatility_targeting**  
   Allocate by risk contribution.

5. **04_06_var_cvar_and_stress_testing**  
   Portfolio tail risk.

6. **05_01_vectorized_backtest_engine**  
   Generalise this accounting into reusable backtesting infrastructure.

## 33. Summary

This notebook implemented a multi-asset return engine.

It showed how to:

1. define an asset master;
2. simulate local prices and FX rates;
3. convert prices to USD;
4. flag and handle missing data;
5. calculate simple and log returns;
6. build return matrices;
7. generate target and investable weights;
8. shift weights to avoid look-ahead;
9. calculate turnover and transaction costs;
10. calculate gross and net portfolio returns;
11. build NAV and dollar PnL;
12. perform asset, class, and currency attribution;
13. diagnose exposure;
14. check futures-style PnL;
15. evaluate stress days;
16. run audit checks;
17. save a reusable return-engine dataset.

The key computational takeaway:

> Return calculation is not a trivial preprocessing step; it is the accounting foundation of every portfolio result.

The key financial takeaway:

> Portfolio returns are only meaningful when price, FX, weight timing, costs, and instrument metadata are aligned explicitly.

## 34. Further reading

- Grinold and Kahn, *Active Portfolio Management.*
- Litterman and modern portfolio construction literature.
- Meucci, *Risk and Asset Allocation.*
- Futures portfolio accounting and continuous contract construction literature.
- Portfolio attribution, transaction-cost modelling, and multi-currency performance measurement references.